# ConvLSTM — Quarterly Resolution

**Architecture 1 of 5** (`01_quarterly_convlstm.ipynb`)

**Stand-alone training notebook.** This file inlines the complete pipeline — configuration, preprocessing, losses/metrics, sequence generation, the strict temporal split, the `ConvLSTM` architecture, training and evaluation — so it can be executed end-to-end without importing any project module.

- **Architecture:** `ConvLSTM` (§5.5, Figure 5.1): A standalone spatiotemporal recurrent network: per-frame convolutional feature extraction followed by stacked ConvLSTM2D layers that model temporal evolution and a lightweight decoder that restores the full resolution.
- **Temporal resolution:** Quarterly (quarter composite)
- **Sequence lengths L:** [6, 8, 10]
- **Temporal split:** Setup 1 (train ≤ 2015 / test 2016–2025) and Setup 2 (train ≤ 2020 / test 2021–2025)
- **Loss:** `L_BCE + L_Dice`; optimiser Adam(lr=1e-4, clipnorm=1.0)
- **Evaluation:** IoU, Dice, Precision, Recall and signed area difference ΔA (km²)

> Best architecture for the quarterly horizon (report §7.2): **Attention U-Net + ConvLSTM**, Setup 1 -- IoU = 0.6956, Dice = 0.8139.


## 1. Configuration and imports

Environment setup, imports, random seeds and the experiment configuration (thesis §5.9).


In [ ]:
# ============================================================================
# Environment, imports and configuration
# ============================================================================
# Everything this notebook needs is defined inline so the file is stand-alone.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("XLA_FLAGS", "--xla_gpu_strict_conv_algorithm_picker=false")

import re
import glob
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import tensorflow as tf
tf.get_logger().setLevel("FATAL")

from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.callbacks import (
    Callback, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau,
)
from tensorflow.keras.optimizers import Adam

import rasterio
import cv2
from skimage.transform import resize
from sklearn.metrics import precision_score, recall_score

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow


def fit_and_track(model, train_data, train_labels, *, config, run_name,
                  validation_data=None, callbacks=(), checkpoint_path=None,
                  enable_autolog=True, **fit_kwargs):
    # Train one configuration and persist its MLflow run and checkpoint.
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "file:./mlruns"))
    mlflow.set_experiment(os.environ.get("MLFLOW_EXPERIMENT_NAME", "river-morphology"))
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({f"config.{key}": str(value) for key, value in config.items()})
        if enable_autolog:
            mlflow.tensorflow.autolog(log_models=False, silent=True)
        if validation_data is not None:
            fit_kwargs["validation_data"] = validation_data
        history = model.fit(train_data, train_labels, callbacks=list(callbacks), **fit_kwargs)
        for metric, values in history.history.items():
            if values:
                mlflow.log_metric(f"final.{metric}", float(values[-1]))
        if checkpoint_path and os.path.isfile(checkpoint_path):
            mlflow.log_artifact(checkpoint_path, artifact_path="checkpoints")
        return history

# ---- Reproducibility -------------------------------------------------------
np.random.seed(42)
tf.random.set_seed(42)
for _gpu in tf.config.experimental.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError:
        pass

# ---- Configuration (thesis section 5.9) ------------------------------------
DATA_DIR         = os.environ.get("QUARTERLY_DIR", os.path.join("data", "raw", "quarterly"))
IMG_HEIGHT       = 256
IMG_WIDTH        = 256
N_COMPONENTS     = 3
BINARY_THRESHOLD = 0.5
LEARNING_RATE    = 1e-4
CLIPNORM         = 1.0
DEFAULT_EPOCHS   = 200
EARLY_STOP_PATIENCE = 20
REDUCE_LR_PATIENCE  = 7
BATCH_SIZE       = 4

RESOLUTION       = "quarterly"
SEQUENCE_LENGTHS = [6, 8, 10]
MODEL_LABEL      = "ConvLSTM"
MODEL_KEY        = "convlstm"
SETUP_CONFIGS = {
    "Setup 1": {"cutoff_year": 2015, "test_label": "2016-2025"},
    "Setup 2": {"cutoff_year": 2020, "test_label": "2021-2025"},
}
OUTPUT_DIR = os.path.join("outputs", RESOLUTION)
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

print("Data directory :", DATA_DIR)
print("Resolution     :", RESOLUTION, "| sequence lengths:", SEQUENCE_LENGTHS)
print("Model          :", MODEL_LABEL, "(", MODEL_KEY, ")")


## 2. Preprocessing, losses, metrics and data pipeline

These helpers are copied verbatim from the reviewed `utils/model_utils.py`, keeping this notebook fully self-contained: connected-component cleaning, water-mask loading, the BCE + Dice loss (equations 5.7–5.9), the IoU/Dice/area metrics (equations 5.2–5.6), sliding-window sequences (§5.1) and the leakage-proof temporal split (§5.2).


In [ ]:
# ============================================================================
# Losses and metrics  (thesis equations 5.6-5.9)
# ============================================================================


def dice_coefficient(y_true, y_pred, smooth: float = 1e-6):
    """Soft Dice coefficient (differentiable) used as a training metric."""
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def dice_loss(y_true, y_pred):
    """Region-based loss: ``1 - Dice`` (equation 5.7 of the thesis)."""
    return 1.0 - dice_coefficient(y_true, y_pred)


def bce_loss(y_true, y_pred):
    """Pixel-wise binary cross-entropy (equation 5.8 of the thesis)."""
    return K.mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))


def combined_loss(y_true, y_pred):
    """
    Hybrid objective ``L_BCE + L_Dice`` (equation 5.9).

    BCE gives stable early gradients; Dice sharpens boundaries.  The sum
    balances pixel-level accuracy with region-level overlap, which is the
    configuration used for *all* five architectures.
    """
    return bce_loss(y_true, y_pred) + dice_loss(y_true, y_pred)


def iou_metric(y_true, y_pred, threshold: float = BINARY_THRESHOLD):
    """Binarised Intersection-over-Union as a Keras metric."""
    y_pred_b = K.cast(y_pred > threshold, "float32")
    y_true_b = K.cast(y_true > threshold, "float32")
    intersection = K.sum(y_true_b * y_pred_b)
    union = K.sum(y_true_b) + K.sum(y_pred_b) - intersection
    return (intersection + K.epsilon()) / (union + K.epsilon())


# ============================================================================
# NumPy metrics used at evaluation time
# ============================================================================


def iou_np(y_true: np.ndarray, y_pred: np.ndarray, threshold: float = BINARY_THRESHOLD) -> float:
    """Intersection-over-Union (Jaccard index) on binarised arrays.

    Parameters
    ----------
    y_true, y_pred : np.ndarray
        Continuous or binary prediction masks of identical shape.
    threshold : float
        Values strictly greater than this threshold count as water (1).

    Returns
    -------
    float
        IoU score in [0, 1] (with a small epsilon for numerical stability).
    """
    yt = (y_true > threshold).astype(np.float32)
    yp = (y_pred > threshold).astype(np.float32)
    inter = np.sum(yt * yp)
    union = np.sum(yt) + np.sum(yp) - inter
    return float(inter / (union + 1e-6))


def dice_np(y_true: np.ndarray, y_pred: np.ndarray, threshold: float = BINARY_THRESHOLD) -> float:
    """F1 / Dice coefficient on binarised arrays.

    Parameters
    ----------
    y_true, y_pred : np.ndarray
        Continuous or binary prediction masks of identical shape.
    threshold : float
        Values strictly greater than this threshold count as water (1).

    Returns
    -------
    float
        Dice score in [0, 1] (with a small epsilon for numerical stability).
    """
    yt = (y_true > threshold).astype(np.float32)
    yp = (y_pred > threshold).astype(np.float32)
    inter = np.sum(yt * yp)
    return float((2.0 * inter) / (np.sum(yt) + np.sum(yp) + 1e-6))


def calculate_area_difference(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    pixel_area_km2: float,
    threshold: float = BINARY_THRESHOLD,
) -> float:
    r"""Signed water-area difference ΔA = A_pred − A_true  (km²).

    A positive value means the prediction overestimates water area;
    a negative value means it underestimates.  Equation 5.6 of the thesis.

    Parameters
    ----------
    y_true, y_pred : np.ndarray
        Masks of identical shape.
    pixel_area_km2 : float
        Ground area covered by one pixel, in km².
    threshold : float
        Binarisation threshold (default 0.5).

    Returns
    -------
    float
        Signed area difference in km².
    """
    true_area = np.sum((y_true > threshold).astype(np.float32)) * pixel_area_km2
    pred_area = np.sum((y_pred > threshold).astype(np.float32)) * pixel_area_km2
    return float(pred_area - true_area)


# ============================================================================
# Data loading and preprocessing
# ============================================================================


def keep_largest_n_components_cv2(mask: np.ndarray, n: int = N_COMPONENTS):
    """Retain only the *n* largest connected components of a binary mask.

    This removes salt-and-pepper noise and isolated ponds so the network
    focuses on the main river channel (thesis section 5.9).
    """
    binary_mask = (mask > 0).astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    if num_labels <= 1:
        return np.zeros_like(mask)

    areas = stats[1:, cv2.CC_STAT_AREA]
    sorted_idx = np.argsort(areas)[::-1]
    n_keep = min(n, len(sorted_idx))

    cleaned = np.zeros_like(mask)
    for idx in sorted_idx[:n_keep]:
        cleaned[labels == (idx + 1)] = mask.max()
    return cleaned


def load_and_preprocess_image(
    filepath: str, apply_cleaning: bool = True, n_components: int = N_COMPONENTS
) -> np.ndarray:
    """Load a GeoTIFF water mask and turn it into a 256x256 clean binary array.

    Steps: read -> optional connected-component cleaning -> resize (nearest)
    -> binarise at :data:`BINARY_THRESHOLD`.
    """
    with rasterio.open(filepath) as src:
        img = src.read(1)
    if apply_cleaning:
        img = keep_largest_n_components_cv2(img, n=n_components)
    resized = resize(
        img,
        (IMG_HEIGHT, IMG_WIDTH),
        mode="constant",
        preserve_range=True,
        anti_aliasing=False,
    )
    return (resized > BINARY_THRESHOLD).astype(np.float32)


def get_pixel_area_km2(reference_tif: str) -> float:
    """Compute the ground area (km2) covered by a single 256x256 pixel.

    Handles both EPSG:4326 (geographic) and projected CRSs.
    """
    with rasterio.open(reference_tif) as src:
        bounds = src.bounds
        if src.crs and src.crs.to_epsg() == 4326:
            centre_lat = (bounds.top + bounds.bottom) / 2
            m_per_deg_lon = 111_320 * np.cos(np.radians(centre_lat))
            width_m = (bounds.right - bounds.left) * m_per_deg_lon
            height_m = (bounds.top - bounds.bottom) * 111_320
            total_km2 = (width_m * height_m) / 1e6
        else:
            total_km2 = ((bounds.right - bounds.left) * (bounds.top - bounds.bottom)) / 1e6
    return total_km2 / (IMG_HEIGHT * IMG_WIDTH)


def build_catalog(data_dir: str, pattern: str = "*.tif") -> pd.DataFrame:
    """Scan *data_dir* for GeoTIFFs and return a year-sorted DataFrame.

    Columns: ``filepath``, ``filename``, ``year``.  Files whose name contains
    no 4-digit year are skipped.

    The column schema is always returned -- even when the directory is empty or
    contains nothing usable -- so callers can rely on ``df["year"]`` without a
    guard clause.
    """
    data_dir = validate_path(data_dir, "data_dir")
    pattern = validate_safe_pattern(pattern)

    files = sorted(glob.glob(os.path.join(str(data_dir), pattern)))
    records = []
    for filepath in files:
        match = re.search(r"(\d{4})", os.path.basename(filepath))
        if match:
            records.append(
                {
                    "filepath": filepath,
                    "filename": os.path.basename(filepath),
                    "year": int(match.group(1)),
                }
            )

    if not records:
        return pd.DataFrame(columns=["filepath", "filename", "year"])

    return pd.DataFrame(records).sort_values("year").reset_index(drop=True)


def load_image_stack(data_dir: str) -> Tuple[List[np.ndarray], List[int]]:
    """Load and preprocess every GeoTIFF in *data_dir* into a stack.

    Raises
    ------
    FileNotFoundError
        If *data_dir* does not exist, or exists but holds no GeoTIFF whose
        filename contains a 4-digit year.  Failing loudly here is deliberate:
        a silently empty stack only surfaces much later as a confusing error
        inside the sequence/split code (or, worse, as a model trained on
        nothing).
    """
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"Data directory does not exist: {data_dir!r}")

    df = build_catalog(data_dir)
    if df.empty:
        raise FileNotFoundError(
            f"No year-tagged GeoTIFFs found in {data_dir!r}. Expected filenames "
            f"containing a 4-digit year, e.g. 'Padma_2015.tif'."
        )

    images, years = [], []
    for _, row in df.iterrows():
        images.append(load_and_preprocess_image(row["filepath"]))
        years.append(int(row["year"]))
    return images, years


# ============================================================================
# Sequence generation  (sliding window, section 5.1)
# ============================================================================


def create_sequences(
    images: Sequence[np.ndarray],
    years: Sequence[int],
    seq_len: int,
    horizon: int = 1,
    stride: int = 1,
):
    """Build overlapping ``(X, y)`` windows with a stride of 1.

    Parameters
    ----------
    images   : list of (H, W) binary frames, chronological order
    years    : matching year labels
    seq_len  : number of input frames ``L``
    horizon  : prediction horizon (thesis uses 1)
    stride   : window stride (thesis uses 1)

    Returns
    -------
    X, y, input_years, target_years : np.ndarrays
    """
    if not isinstance(seq_len, int) or isinstance(seq_len, bool) or seq_len < 1:
        raise ValueError("seq_len must be a positive integer")
    if not isinstance(horizon, int) or isinstance(horizon, bool) or horizon < 1:
        raise ValueError("horizon must be a positive integer")
    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 1:
        raise ValueError("stride must be a positive integer")
    if len(images) != len(years):
        raise ValueError("images and years must have the same length")

    X, y, in_years, tgt_years = [], [], [], []
    stop = len(images) - seq_len - horizon + 1
    for i in range(0, stop, stride):
        X.append(images[i : i + seq_len])
        y.append(images[i + seq_len + horizon - 1])
        in_years.append(years[i : i + seq_len])
        tgt_years.append(years[i + seq_len + horizon - 1])
    return (np.array(X), np.array(y), np.array(in_years), np.array(tgt_years))


# ============================================================================
# Strict temporal split  (section 5.2)
# ============================================================================


def prepare_split(X_all, y_all, target_years_all, input_years_all, cutoff_year: int):
    """Leakage-proof temporal split.

    Training/validation use samples whose *target* and *latest input* years
    are <= ``cutoff_year``; testing uses targets strictly after the cutoff.
    The last 15 % of the (chronologically ordered) training samples become
    the validation set.

    Returns
    -------
    X_tr, y_tr, X_val, y_val, X_test, y_test, target_years_test
    (all arrays already expanded with a trailing channel axis)
    """
    max_input_year = np.array([iy.max() for iy in input_years_all])

    train_mask = (target_years_all <= cutoff_year) & (max_input_year <= cutoff_year)
    test_mask = target_years_all > cutoff_year

    X_train, y_train = X_all[train_mask], y_all[train_mask]
    ty_train = target_years_all[train_mask]
    X_test, y_test = X_all[test_mask], y_all[test_mask]
    ty_test = target_years_all[test_mask]

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError(
            f"Empty split! Train={len(X_train)}, Test={len(X_test)} (cutoff={cutoff_year})"
        )

    order = np.argsort(ty_train)
    X_train, y_train, ty_train = X_train[order], y_train[order], ty_train[order]

    split = int(len(X_train) * 0.85)
    X_tr, y_tr = X_train[:split], y_train[:split]
    X_val, y_val = X_train[split:], y_train[split:]

    if len(X_val) == 0:
        raise ValueError("Empty validation set!")

    overlap = set(ty_train.tolist()) & set(ty_test.tolist())
    assert not overlap, f"DATA LEAK! Overlapping years: {overlap}"

    def _add_channel(a):
        return np.expand_dims(a, axis=-1)

    return (
        _add_channel(X_tr),
        _add_channel(y_tr),
        _add_channel(X_val),
        _add_channel(y_val),
        _add_channel(X_test),
        _add_channel(y_test),
        ty_test,
    )


# ============================================================================
# Evaluation helpers
# ============================================================================


def evaluate_model(
    model,
    X_test,
    y_test,
    target_years_test,
    pixel_area_km2: float,
    model_label: str,
    setup_name: str,
    include_persistence: bool = True,
) -> pd.DataFrame:
    """
    Evaluate a trained model (and optionally a persistence baseline) on the
    test set, returning a tidy per-sample DataFrame with IoU, Dice,
    Precision, Recall and signed area difference.
    """
    pred = model.predict(X_test, verbose=0)
    comparisons = [(model_label, pred)]
    if include_persistence:
        comparisons.append(("Persistence", X_test[:, -1, :, :, :]))

    rows = []
    for name, predictions in comparisons:
        for i in range(len(y_test)):
            yt = y_test[i, :, :, 0]
            yp = predictions[i, :, :, 0]
            yt_flat = (yt.flatten() > BINARY_THRESHOLD).astype(int)
            yp_flat = (yp.flatten() > BINARY_THRESHOLD).astype(int)
            rows.append(
                {
                    "Setup": setup_name,
                    "Model": name,
                    "Year": target_years_test[i],
                    "IoU": iou_np(yt, yp),
                    "Dice": dice_np(yt, yp),
                    "Precision": precision_score(yt_flat, yp_flat, zero_division=0),
                    "Recall": recall_score(yt_flat, yp_flat, zero_division=0),
                    "Area_Diff_km2": calculate_area_difference(yt, yp, pixel_area_km2),
                }
            )
    return pd.DataFrame(rows)


def summarise_results(df_results: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-sample results into per (Setup, Model) means."""
    cols = ["IoU", "Dice", "Precision", "Recall", "Area_Diff_km2"]
    agg = df_results.groupby(["Setup", "Model"])[cols].mean().round(4)
    agg["Abs_Area_Diff_km2"] = (
        df_results.groupby(["Setup", "Model"])["Area_Diff_km2"]
        .apply(lambda s: s.abs().mean())
        .round(4)
    )
    return agg.reset_index()


## 3. Load the quarterly water-mask time series

Every GeoTIFF in the data directory is read, cleaned and resized to 256×256 binary masks.


In [ ]:
# ============================================================================
# Load the quarterly water-mask time series
# ============================================================================
images, years = load_image_stack(DATA_DIR)
print(f"Loaded {len(images)} quarter frames: {years[0]} ... {years[-1]}")
print("Frame shape :", images[0].shape)

# Ground area of one 256x256 pixel (km2), derived from the GeoTIFF georeferencing
PIXEL_AREA_KM2 = get_pixel_area_km2(build_catalog(DATA_DIR).iloc[0]["filepath"])
print(f"Pixel ground area : {PIXEL_AREA_KM2:.6f} km2")


## 4. Model architecture — ConvLSTM

Definition of the ConvLSTM network (§5.5, Figure 5.1), inlined below.


In [ ]:
# ============================================================================
# Model architecture -- ConvLSTM   (report §5.5, Figure 5.1)
# ============================================================================
# ============================================================================
# Optimiser  (thesis section 5.9)
# ============================================================================


def _adam():
    return Adam(learning_rate=LEARNING_RATE, clipnorm=CLIPNORM)


def build_convlstm(seq_len: int, input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)):
    """Architecture 1 – standalone ConvLSTM (thesis Figure 5.1)."""
    inputs = layers.Input(shape=(seq_len, *input_shape))

    x = layers.TimeDistributed(layers.Conv2D(32, 3, padding="same", activation="relu"))(inputs)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D(2))(x)
    x = layers.TimeDistributed(layers.Dropout(0.2))(x)

    x = layers.TimeDistributed(layers.Conv2D(64, 3, padding="same", activation="relu"))(x)
    x = layers.TimeDistributed(layers.BatchNormalization())(x)
    x = layers.TimeDistributed(layers.MaxPooling2D(2))(x)
    x = layers.TimeDistributed(layers.Dropout(0.2))(x)

    x = layers.ConvLSTM2D(
        128,
        (3, 3),
        padding="same",
        return_sequences=True,
        dropout=0.3,
        kernel_regularizer=tf.keras.regularizers.l2(1e-3),
    )(x)
    x = layers.BatchNormalization()(x)

    x = layers.ConvLSTM2D(
        64,
        (3, 3),
        padding="same",
        return_sequences=False,
        dropout=0.3,
        kernel_regularizer=tf.keras.regularizers.l2(1e-3),
    )(x)
    x = layers.BatchNormalization()(x)

    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.UpSampling2D(2)(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv2D(16, 3, padding="same", activation="relu")(x)
    outputs = layers.Conv2D(1, 1, padding="same", activation="sigmoid")(x)

    model = models.Model(inputs, outputs, name=f"ConvLSTM_seq{seq_len}")
    model.compile(optimizer=_adam(), loss=combined_loss, metrics=[dice_coefficient, iou_metric])
    return model

# Sanity check: build the network for the first sequence length.
build_convlstm(SEQUENCE_LENGTHS[0]).summary()


## 5. Training across sequence lengths and temporal setups

The architecture is trained for every sequence length [6, 8, 10] under both temporal setups, with `ModelCheckpoint`, `EarlyStopping` and `ReduceLROnPlateau`.


In [ ]:
# ============================================================================
# Training callbacks and the multi-configuration training loop
# ============================================================================
class StructuredTrainingLogger(Callback):
    """Compact, aligned one-line-per-epoch training logger.

    Emits through the :mod:`logging` framework (logger ``utils.model_utils``)
    instead of ``print`` so that long training runs can be captured, filtered
    and timestamped by the caller's logging configuration.
    """

    def __init__(self, total_epochs: int):
        super().__init__()
        self.total_epochs = total_epochs
        self.best_val_loss = np.inf

    def on_train_begin(self, logs=None):
        header = (
            f"{'Ep':>4s}/{'Tot':<4s} | {'Loss':>8s} | {'VLoss':>8s} | "
            f"{'Dice':>6s} | {'VDice':>6s} | {'IoU':>6s} | "
            f"{'VIoU':>6s} | {'LR':>9s} | Note"
        )
        logger.info("-" * len(header))
        logger.info(header)
        logger.info("-" * len(header))

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        vloss = logs.get("val_loss", 0)
        lr = float(K.get_value(self.model.optimizer.learning_rate))
        note = ""
        if vloss < self.best_val_loss:
            self.best_val_loss = vloss
            note = "saved"
        logger.info(
            "Ep %4d/%-4d | %8.4f | %8.4f | %6.4f | %6.4f | %6.4f | %6.4f | %9.2e | %s",
            epoch + 1,
            self.total_epochs,
            logs.get("loss", 0),
            vloss,
            logs.get("dice_coefficient", 0),
            logs.get("val_dice_coefficient", 0),
            logs.get("iou_metric", 0),
            logs.get("val_iou_metric", 0),
            lr,
            note,
        )

    def on_train_end(self, logs=None):
        logger.info("-" * 85)
        logger.info("  Training finished  |  Best val_loss: %.4f", self.best_val_loss)
        logger.info("-" * 85)


def create_callbacks(
    model_name: str,
    checkpoint_dir: str = "checkpoints",
    epochs: int = DEFAULT_EPOCHS,
    patience: int = EARLY_STOP_PATIENCE,
):
    """Standard callback set (checkpoint, early stop, LR schedule, logger)."""
    os.makedirs(checkpoint_dir, exist_ok=True)
    ckpt_path = os.path.join(checkpoint_dir, f"{model_name}_best.keras")
    return [
        ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, mode="min", verbose=0),
        EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=REDUCE_LR_PATIENCE, min_lr=1e-7, verbose=0
        ),
        StructuredTrainingLogger(total_epochs=epochs),
    ]


# ---- Train every (setup, sequence-length) combination ----------------------
all_results = []
histories = {}

for setup_name, cfg in SETUP_CONFIGS.items():
    for seq_len in SEQUENCE_LENGTHS:
        tag = f"{MODEL_KEY}_L{seq_len}_{setup_name.replace(' ', '')}"
        print()
        print("=" * 78)
        print(f"  {MODEL_LABEL} | {RESOLUTION} | {setup_name} | L={seq_len}")
        print("=" * 78)

        X_all, y_all, in_years, tgt_years = create_sequences(images, years, seq_len)
        X_tr, y_tr, X_val, y_val, X_test, y_test, ty_test = prepare_split(
            X_all, y_all, tgt_years, in_years, cfg["cutoff_year"])
        print(f"  windows -> train={len(X_tr)}  val={len(X_val)}  test={len(X_test)}")

        tf.keras.backend.clear_session()
        model = build_convlstm(seq_len)
        hist = fit_and_track(
            model,
            X_tr,
            y_tr,
            config=cfg,
            run_name=tag,
            validation_data=(X_val, y_val),
            checkpoint_path=os.path.join(CKPT_DIR, f"{tag}_best.keras"),
            epochs=DEFAULT_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=create_callbacks(tag, checkpoint_dir=CKPT_DIR),
            enable_autolog=True,
            verbose=0,
        )
        histories[(setup_name, seq_len)] = hist

        df_eval = evaluate_model(
            model, X_test, y_test, ty_test,
            PIXEL_AREA_KM2, MODEL_LABEL, setup_name)
        df_eval["L"] = seq_len
        all_results.append(df_eval)

df_all = pd.concat(all_results, ignore_index=True)
df_summary = summarise_results(df_all)
df_by_len = (df_all.groupby(["Setup", "L"])
             [["IoU", "Dice", "Precision", "Recall", "Area_Diff_km2"]]
             .mean().round(4).reset_index())

print()
print("=== Mean metrics per (setup, sequence length) ===")
df_by_len


## 6. Comparative summary


In [ ]:
# ============================================================================
# Comparative summary (mean over all test windows)
# ============================================================================
print("Mean metrics per setup (compare with docs/predefence_report.md section 7):")
df_summary


## 7. Visualisations


In [ ]:
# ============================================================================
# Visualisations: training curves and sequence-length comparison
# ============================================================================
# -- 1. Training / validation loss curves ------------------------------------
n = len(histories)
fig, axes = plt.subplots(n, 1, figsize=(9, 2.6 * n), squeeze=False)
for ax, ((setup_name, seq_len), hist) in zip(axes[:, 0], histories.items()):
    ax.plot(hist.history["loss"], label="train")
    ax.plot(hist.history["val_loss"], label="validation")
    ax.set_title(f"{setup_name} | L={seq_len} | loss")
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"{MODEL_KEY}_training_curves.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# -- 2. IoU vs sequence length for each setup --------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
for setup_name in df_by_len["Setup"].unique():
    sub = df_by_len[df_by_len["Setup"] == setup_name]
    ax.plot(sub["L"], sub["IoU"], marker="o", label=setup_name)
ax.set_xlabel("sequence length L")
ax.set_ylabel("mean IoU")
ax.set_title(f"{MODEL_LABEL} -- {RESOLUTION}: IoU vs L")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"{MODEL_KEY}_iou_vs_L.png"),
            dpi=150, bbox_inches="tight")
plt.show()
